# **Entreno de ángulos mediante el algoritmo de Ben Priestley**

En la siguiente sección voy a realizar un preentreno de ángulos para instancias pequeñas de CVP basados en la factorización de N

In [44]:
%load_ext autoreload
%autoreload 2

import time
import sympy
import numpy as np
from math import log, log2

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from Crypto.Util import number


from tqdm.notebook import tqdm

import random

from scipy.optimize import curve_fit, least_squares

#Importar de función interna
from modules import schnorr_lattice as sl
from modules import qaoa
from modules import utils

from modules.functions import solve_cvp, solve_cvp_with_opt_paramters

from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
seed = 42
random.seed(seed)

In [46]:
def generate_N(bitLength):
    """
    param bitLength: cantidad de bits de N

    return: N = p*q de bitLength bits
    """
    
    min_p_bit_length = max(2, bitLength // 4)
    max_p_bit_length = bitLength - min_p_bit_length

    

    while True:
        p_bit_length = random.randint(min_p_bit_length, max_p_bit_length)

        p = number.getPrime(p_bit_length)

        q_bit_length = bitLength - p_bit_length + 1

        q = number.getPrime(q_bit_length)

        N = p * q

        if N.bit_length() == bitLength:
            return N


## **Generación de set de entrenamiento**

Código de generación del set de entrenamiento

In [47]:
"""
training_set = []
tSet_length = 300
min_train_bitLength = 8
max_train_bitLength = 128

for _ in range(tSet_length):
    bitLength = random.randint(min_train_bitLength, max_train_bitLength)
    N = generate_N(bitLength)
    training_set.append(N)


train_df = pd.DataFrame({
    'N': training_set,
    'Bit_length': [n.bit_length() for n in training_set]
})


train_df.to_csv(f'./training_sets/training_set_size300_randomBitLength_8_to_128.csv')   """ 

"\ntraining_set = []\ntSet_length = 300\nmin_train_bitLength = 8\nmax_train_bitLength = 128\n\nfor _ in range(tSet_length):\n    bitLength = random.randint(min_train_bitLength, max_train_bitLength)\n    N = generate_N(bitLength)\n    training_set.append(N)\n\n\ntrain_df = pd.DataFrame({\n    'N': training_set,\n    'Bit_length': [n.bit_length() for n in training_set]\n})\n\n\ntrain_df.to_csv(f'./training_sets/training_set_size300_randomBitLength_8_to_128.csv')   "

## **Generación de set de validación**

Código de generación del set de validación

In [48]:

""" validation_set = []
min_val_bitLength = 8
max_val_bitLength = 128
n_per_length = 1

for length in range(min_val_bitLength, max_val_bitLength):
    for _ in range(n_per_length):
        N = generate_N(length)
        validation_set.append(N)

val_df = pd.DataFrame({
    'N': validation_set,
    'bitLength': [n.bit_length() for n in validation_set]
})


val_df.to_csv(f'./validation_sets/validation_set_size120_bitLength_8_to_128.csv') """


" validation_set = []\nmin_val_bitLength = 8\nmax_val_bitLength = 128\nn_per_length = 1\n\nfor length in range(min_val_bitLength, max_val_bitLength):\n    for _ in range(n_per_length):\n        N = generate_N(length)\n        validation_set.append(N)\n\nval_df = pd.DataFrame({\n    'N': validation_set,\n    'bitLength': [n.bit_length() for n in validation_set]\n})\n\n\nval_df.to_csv(f'./validation_sets/validation_set_size120_bitLength_8_to_128.csv') "

## **Proceso de entrenamiento**

In [49]:
#Función para ver el escalado de la probabilidad de obtener una mejor solución del paper de Ben Priestley
def scaling_function(x, alpha):
    return 1/(2**(alpha*x))

In [50]:
def normalize_angles(angles: list[list[float]]):
    #Normalizo los angulos en el intervalo: [-π, π]
    nangles = np.array(angles)
    nangles = (nangles + np.pi) % (2*np.pi) - np.pi
    return nangles


def obtain_kmeans(k: int, angles: np.ndarray) -> list[list[float]]:
    kmeans = KMeans(n_clusters = k, random_state = seed, n_init="auto")

    n_angles = angles.shape[1]
    features = np.hstack([
        np.column_stack([np.cos(angles[:, i]), np.sin(angles[:, i])])
        for i in range(n_angles)
    ])

    kmeans.fit(features)
    c = kmeans.cluster_centers_
    centroides = np.column_stack([
        np.arctan2(c[:, 2*i + 1], c[:, 2*i])
        for i in range(n_angles)
    ])

    return centroides.tolist()

In [51]:
def qaoa_pretrain_algorithm(train_set: np.ndarray, val_set: np.ndarray, epochs: int = 5, clusters: int = 10,
                            delta: float = 0.75, l: int = 1, c: int = 3, min_method: str = 'Nelder-Mead', p: int = 1,
                            shots: int = 10_000, q: int = 10,
                            seed: int = 42, verbose: bool = False, progress_bar: bool = False):
    """
    TODO
    """
    # Inicializacion
    np.random.seed(seed)
    a_op = None #Asignación de angulos inicial
    best_alpha = np.inf
    val_stats = {}

    #train_len = train_set.size
    #val_len = val_set.size

    
    for epoch_idx in tqdm(range(1, epochs + 1), desc = "Epochs") if progress_bar else range(1, epochs + 1): #Episodios de entreno

        tic = time.perf_counter()


        angles_population = [] #Conjunto con los angulos calculados para cada instancia de entreno
        angles_clusters_rep = [] #Conjunto con los representantes de los clusters 

        #Entrenamiento
        for tN in tqdm(train_set, desc = "Training") if progress_bar else train_set: 
            
            cvp = sl.schnorrCVP(tN, c, l, seed, verbose = False)#Genero una instancia para realizar los calculos
            instance = cvp.generate_cvp(q, verbose = False)

            if a_op is not None: x0 = np.asarray(a_op)
            else: x0 = np.asarray([0.0]*(2*p))

            #Obtengo los angulos optimos
            _,_,_, angles = solve_cvp(cvp, instance, x0, delta, shots = 1, p = p, min_method= min_method)

            angles_population.append(list(angles.values()))
        #Fin entrenamiento
        
        #Division de Clusters y sus representantes
        #TODO
        normalized_angles = normalize_angles(angles_population)

        angles_clusters_rep = obtain_kmeans(clusters, normalized_angles)
        #---------------------------------------


        #Validacion
        alphas = []
        for a in tqdm(angles_clusters_rep, desc = "Optimal Angles") if progress_bar else angles_clusters_rep: #Por cada angulo representante de su cluster
            lattice_dimension = []
            probabilities = []


            for vN in tqdm(val_set, desc = "Validation") if progress_bar else val_set: #Por cada instancia de validacion
                
                cvp = sl.schnorrCVP(vN, c, l, seed, verbose = False)
                instance = cvp.generate_cvp(q, verbose = False)

                vnews, probs, _ , _ = solve_cvp_with_opt_paramters(cvp, instance, a, delta, shots, p)
                dists2 = utils.get_distances2(vnews, instance.t)
                
                best_dist = np.inf
                best_prob = 0.0

                for d, prob in zip(dists2, probs):
                    if d < best_dist:
                        best_dist = d
                        best_prob = prob

                lattice_dimension.append(cvp.n)
                probabilities.append(best_prob)
                
                
            alpha = curve_fit(scaling_function, xdata = lattice_dimension, ydata = probabilities)[0][0]
            alphas.append(alpha)
        #Fin de validacion

        tac = time.perf_counter()

        # Epoch statistics.
        min_alpha = min(alphas)
        mean_alpha = np.mean(alphas)
        std_alpha = np.std(alphas)
        max_alpha = max(alphas)
        runtime = tac - tic
        epoch_stats = [min_alpha, mean_alpha, std_alpha, max_alpha, runtime]
        val_stats[epoch_idx] = epoch_stats

        if verbose:
            print(f'Epoch {epoch_idx:02d} ({int(runtime):04d} s): min - {min_alpha:.3f}, mean - {mean_alpha:.3f}, std - {std_alpha:.3f}, max - {max_alpha:.3f}')

        if min_alpha < best_alpha:
            best_alpha = min_alpha
            a_op = angles_clusters_rep[np.argmin(alphas)]

    return a_op, best_alpha, val_stats
    

### **Entrenamiento 1**

In [52]:
""" training_set = []
tSet_length = 300
min_train_bitLength = 8
max_train_bitLength = 128

for _ in range(tSet_length):
    bitLength = random.randint(min_train_bitLength, max_train_bitLength)
    N = generate_N(bitLength)
    training_set.append(N)


train_df = pd.DataFrame({
    'N': training_set,
    'Bit_length': [n.bit_length() for n in training_set]
})


train_df.to_csv(f'./training_sets/training_set_size300_randomBitLength_8_to_128.csv', index = False)    """

" training_set = []\ntSet_length = 300\nmin_train_bitLength = 8\nmax_train_bitLength = 128\n\nfor _ in range(tSet_length):\n    bitLength = random.randint(min_train_bitLength, max_train_bitLength)\n    N = generate_N(bitLength)\n    training_set.append(N)\n\n\ntrain_df = pd.DataFrame({\n    'N': training_set,\n    'Bit_length': [n.bit_length() for n in training_set]\n})\n\n\ntrain_df.to_csv(f'./training_sets/training_set_size300_randomBitLength_8_to_128.csv', index = False)    "

In [53]:
"""validation_set = []
min_val_bitLength = 8
max_val_bitLength = 128
n_per_length = 1

for length in range(min_val_bitLength, max_val_bitLength):
    for _ in range(n_per_length):
        N = generate_N(length)
        validation_set.append(N)

val_df = pd.DataFrame({
    'N': validation_set,
    'bitLength': [n.bit_length() for n in validation_set]
})


val_df.to_csv(f'./validation_sets/validation_set_size120_bitLength_8_to_128.csv', index = False)"""

"validation_set = []\nmin_val_bitLength = 8\nmax_val_bitLength = 128\nn_per_length = 1\n\nfor length in range(min_val_bitLength, max_val_bitLength):\n    for _ in range(n_per_length):\n        N = generate_N(length)\n        validation_set.append(N)\n\nval_df = pd.DataFrame({\n    'N': validation_set,\n    'bitLength': [n.bit_length() for n in validation_set]\n})\n\n\nval_df.to_csv(f'./validation_sets/validation_set_size120_bitLength_8_to_128.csv', index = False)"

In [54]:
train_df = pd.read_csv("./training_sets/training_set_size300_randomBitLength_8_to_128.csv")
val_df = pd.read_csv("./validation_sets/validation_set_size120_bitLength_8_to_128.csv")

In [55]:
train_set = train_df["N"].to_numpy()
val_set = val_df["N"].to_numpy()

In [56]:
val_stats_list = []
optimal_assignments = []
best_alphas = {}
for p in range(1, 11):
    print(f"Para el numero de capas p = {p}")
    
    optimal_params, best_alpha, val_stats = qaoa_pretrain_algorithm(train_set, val_set, epochs= 5, clusters = 10,
                                                                     delta = 0.75, l = 1, c = 3, p = p, shots = 10_000, seed = 42, verbose = True)
    
    for epoch_idx, epoch_stats in val_stats.items():
        val_stats_list.append([p, epoch_idx] + epoch_stats)
    optimal_assignments.append(optimal_params)
    best_alphas[f"{p}"] = best_alpha
    print(f"Los parametros mas optimos proporcionan un escalado de {best_alpha}")


Para el numero de capas p = 1
Epoch 01 (1155 s): min - 0.526, mean - 0.568, std - 0.030, max - 0.635
Epoch 02 (0509 s): min - 0.528, mean - 0.572, std - 0.037, max - 0.656
Epoch 03 (0511 s): min - 0.526, mean - 0.572, std - 0.037, max - 0.655
Epoch 04 (0510 s): min - 0.525, mean - 0.566, std - 0.037, max - 0.647
Epoch 05 (0634 s): min - 0.525, mean - 0.567, std - 0.038, max - 0.651
Los parametros mas optimos proporcionan un escalado de 0.5248672258036771
Para el numero de capas p = 2
Epoch 01 (4293 s): min - 0.359, mean - 0.649, std - 0.330, max - 1.609
Epoch 02 (1573 s): min - 0.362, mean - 0.634, std - 0.300, max - 1.283
Epoch 03 (1586 s): min - 0.362, mean - 0.634, std - 0.299, max - 1.272
Epoch 04 (1612 s): min - 0.361, mean - 0.634, std - 0.300, max - 1.279
Epoch 05 (1598 s): min - 0.362, mean - 0.634, std - 0.300, max - 1.281
Los parametros mas optimos proporcionan un escalado de 0.35933034598262403
Para el numero de capas p = 3
Epoch 01 (10116 s): min - 0.366, mean - 0.736, std 

KeyboardInterrupt: 

In [57]:
val_stats_df = pd.DataFrame(val_stats_list, columns = ['p', 'epoch', 'min_alpha', 'mean_alpha', 'std_alpha', 'max_alpha', 'runtime'])
best_alphas_df = pd.DataFrame.from_dict(best_alphas, orient = "index", columns=["best_alphas"])
best_alphas_df.index.name = "p"

In [58]:
val_stats_df = pd.DataFrame(val_stats_list, columns = ['p', 'epoch', 'min_alpha', 'mean_alpha', 'std_alpha', 'max_alpha', 'runtime'])
val_stats_df.to_csv(f"./results/validation_statistics/train300_val120_1.csv", index = False)
best_alphas_df = pd.DataFrame.from_dict(best_alphas, orient = "index", columns=["best_alphas"])
best_alphas_df.index.name = "p"
best_alphas_df.to_csv("./results/training_best_alphas/train300_val120_1.csv")

for i, opt_angles in enumerate(optimal_assignments):
    aux_dict = {}
    for j in range(i + 1):
        aux_dict[f"beta_{j}"] = opt_angles[j]
    for j in range(i + 1):
        aux_dict[f"gamma_{j}"] = opt_angles[j + i + 1]
    
    optimal_assignments_df = pd.DataFrame.from_dict(aux_dict, orient = "index", columns = ["vals"])
    optimal_assignments_df.index.name = "angles"
    optimal_assignments_df.to_csv(f"./results/optimal_angles/train300_val120_1_p{i+1}.csv")

    with open(f"./results/optimal_angles/train300_val120_1_p{i+1}.pkl", "wb") as f:
        pickle.dump(aux_dict, f)
